# Stream A: NLP Fraud Detection — BERT Training
## Guardian Recruit · `02_nlp_stream_training.ipynb`

**Purpose:** Official training notebook for Stream A. Fine-tunes `bert-base-uncased` on the augmented training dataset and exports `models/nlp_bert.pth`.

> **Before running:** In Colab, go to **Runtime → Change runtime type → T4 GPU**. Training on CPU will take several hours.

### Team Roles
| Member | Role |
|--------|------|
| **Srijitha** | BERT implementation, training loop, mixed-precision training |
| **Hemanth** | Data preprocessing, text cleaning, dataset preparation |
| **Isagani Julian** | Evaluation, augmented data integration, model export |
| **Kusuma** | Stream B outlier modeling (`03_outlier_optimization.ipynb`) |

### Hyperparameter Selection
| Param | Value | Justification |
|---|---|---|
| Learning Rate | `2e-5` | Standard BERT sweet spot; `5e-5` caused instability |
| Epochs | `3` | Steady loss reduction without overfitting |
| Batch Size | `16` | Balances GPU memory and gradient stability |
| Max Length | `128` | Covers most posting content; `256` doubles memory |
| Weight Decay | `0.01` | L2 regularisation against minority-class overfitting |
| Warmup | 10% of steps | Prevents large gradients destabilising pre-trained weights |

In [ ]:
# ── GPU check — run this first ───────────────────────────────────────────────
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
    print('In Colab: Runtime → Change runtime type → T4 GPU, then reconnect.')
else:
    print(f'GPU ready: {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pip install transformers scikit-learn matplotlib seaborn --quiet

import pandas as pd
import numpy as np
import re
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm           # tqdm.auto gives clean notebook progress bars
from pathlib import Path

# torch.amp API changed in PyTorch 2.3 — handle both versions
try:
    from torch.amp import autocast, GradScaler
    _AMP_DEVICE = 'cuda'
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    _AMP_DEVICE = None

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'PyTorch {torch.__version__}  |  Device: {device}')

---
## Step 1 — Mount Drive & Upload Augmented Data

The cell below mounts your Drive and sets all file paths automatically.

**If running on Colab for the first time**, run the upload cell after this one to copy `FINAL_AUGMENTED_TRAINING.csv` to your Drive. If it already exists on Drive from a previous run, the upload cell will skip it automatically.

In [ ]:
# ── Environment detection: Colab vs local ────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/DTSC 5082- Group 12 - Guardian Recruit')
    DATA_DIR   = DRIVE_ROOT / 'data' / 'preprocessed'
    MODEL_DIR  = DRIVE_ROOT / 'models'
    IN_COLAB   = True
except ModuleNotFoundError:
    # Resolve repo root relative to this notebook (notebooks/ → ../)
    # Works for any team member regardless of where the repo is cloned.
    REPO_ROOT  = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_DIR   = REPO_ROOT / 'data' / 'processed'
    MODEL_DIR  = REPO_ROOT / 'models'
    IN_COLAB   = False

MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = MODEL_DIR / 'nlp_bert.pth'

print(f'Environment : {"Google Colab" if IN_COLAB else "Local"}')
print(f'Repo root   : {REPO_ROOT if not IN_COLAB else DRIVE_ROOT}')
print(f'Data dir    : {DATA_DIR}')
print(f'Model save  : {MODEL_SAVE_PATH}')

In [ ]:
# ── Upload FINAL_AUGMENTED_TRAINING.csv to Drive (Colab only) ────────────────
# Only runs if the file does not already exist on Drive.
# Skip this cell if you are running locally.

if IN_COLAB:
    aug_drive_path = DATA_DIR / 'FINAL_AUGMENTED_TRAINING.csv'
    if aug_drive_path.exists():
        print(f'Augmented dataset already on Drive: {aug_drive_path}')
        print(f'Rows: {pd.read_csv(aug_drive_path).shape[0]:,}')
    else:
        print('Augmented dataset not found on Drive.')
        print('Upload FINAL_AUGMENTED_TRAINING.csv from your local machine:')
        print('  Local path: data/processed/FINAL_AUGMENTED_TRAINING.csv')
        print()
        from google.colab import files
        uploaded = files.upload()   # opens file picker
        for fname in uploaded:
            dest = DATA_DIR / 'FINAL_AUGMENTED_TRAINING.csv'
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(uploaded[fname])
            print(f'Saved → {dest}')
else:
    print('Local run — skipping Drive upload.')

In [ ]:
# ── Load training data (augmented preferred, original fallback) ──────────────
augmented_path = DATA_DIR / 'FINAL_AUGMENTED_TRAINING.csv'
original_path  = DATA_DIR / 'train_clean_v1.csv'

if augmented_path.exists():
    train_df = pd.read_csv(augmented_path)
    print(f'[OK] Augmented dataset loaded: {augmented_path.name}')
else:
    train_df = pd.read_csv(original_path)
    print(f'[FALLBACK] Augmented file not found — using: {original_path.name}')

val_df = pd.read_csv(DATA_DIR / 'val.csv')

# val.csv stores binary columns as 't'/'f' strings — convert to 0/1
BOOL_MAP = {'t': 1, 'true': 1, 'f': 0, 'false': 0}
for col in ['has_company_logo', 'telecommuting', 'fraudulent', 'has_questions']:
    if col in val_df.columns:
        val_df[col] = (
            val_df[col].astype(str).str.lower()
            .map(BOOL_MAP)
            .fillna(pd.to_numeric(val_df[col], errors='coerce'))
        )

train_df = train_df.dropna(subset=['fraudulent'])
train_df['fraudulent'] = train_df['fraudulent'].astype(int)
val_df['fraudulent']   = val_df['fraudulent'].astype(int)

print(f'Train: {train_df.shape}  |  fraud rate: {train_df["fraudulent"].mean():.2%}')
print(f'Val  : {val_df.shape}    |  fraud rate: {val_df["fraudulent"].mean():.2%}')

In [ ]:
# ── Text field preparation ───────────────────────────────────────────────────
TEXT_COLS = ['title', 'company_profile', 'description', 'requirements']

for col in TEXT_COLS:
    train_df[col] = train_df[col].fillna('')
    val_df[col]   = val_df[col].fillna('')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

for col in TEXT_COLS:
    train_df[col] = train_df[col].apply(clean_text)
    val_df[col]   = val_df[col].apply(clean_text)

train_df['text'] = (train_df['title'] + ' ' + train_df['company_profile'] + ' ' +
                    train_df['description'] + ' ' + train_df['requirements'])
val_df['text']   = (val_df['title'] + ' ' + val_df['company_profile'] + ' ' +
                    val_df['description'] + ' ' + val_df['requirements'])

print('Sample fraud posting (first 300 chars):')
print(train_df[train_df['fraudulent'] == 1]['text'].iloc[0][:300], '...')

In [ ]:
# ── Class weights — computed on full training set ────────────────────────────
print(f'Training on {len(train_df):,} rows  |  fraud rate: {train_df["fraudulent"].mean():.2%}')

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['fraudulent']),
    y=train_df['fraudulent']
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f'Class weights — Legitimate: {class_weights[0]:.4f}  |  Fraudulent: {class_weights[1]:.4f}')
print(f'Missing a fraud penalised {class_weights[1]/class_weights[0]:.1f}x more than a false alarm.')

In [ ]:
# ── Tokeniser & Dataset ──────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128)

class JobDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

print('Tokenising training set...')
train_enc = tokenize(train_df['text'].tolist())
print('Tokenising validation set...')
val_enc   = tokenize(val_df['text'].tolist())

train_dataset = JobDataset(train_enc, train_df['fraudulent'].tolist())
val_dataset   = JobDataset(val_enc,   val_df['fraudulent'].tolist())

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

In [ ]:
# ── Model, optimiser, scheduler ─────────────────────────────────────────────
EPOCHS = 3
LR     = 2e-5

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

# GradScaler: API differs between PyTorch versions
if _AMP_DEVICE:                      # PyTorch >= 2.3
    scaler = GradScaler(_AMP_DEVICE)
else:                                # PyTorch < 2.3
    scaler = GradScaler()

print(f'Model parameters : {sum(p.numel() for p in model.parameters()):,}')
print(f'Epochs           : {EPOCHS}  |  LR: {LR}')
print(f'Total steps      : {total_steps}  |  Warmup: {warmup_steps}')

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────
train_losses = []
loss_fct     = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        if _AMP_DEVICE:                                    # GPU path
            with autocast(_AMP_DEVICE):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss   = loss_fct(logits, labels)
        else:                                              # CPU / old API
            with autocast():
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss   = loss_fct(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'  Epoch {epoch+1} — Avg Training Loss: {avg_loss:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS+1), train_losses, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('Avg Training Loss')
plt.title('Training Loss per Epoch — Stream A BERT')
plt.xticks(range(1, EPOCHS+1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save trained model ───────────────────────────────────────────────────────
torch.save(model.state_dict(), MODEL_SAVE_PATH)
size_mb = MODEL_SAVE_PATH.stat().st_size / 1024 / 1024
print(f'[OK] Model saved → {MODEL_SAVE_PATH}')
print(f'     Size: {size_mb:.1f} MB')
print()
print('To use in inference:')
print('  from nlp_stream import predict_proba_from_row')
print('  prob = predict_proba_from_row(row)  # returns float 0.0-1.0')

---
## Evaluation *(Isagani)*

Evaluating on the held-out **validation set** — data the model never saw during training.

**Why not accuracy?** At 95% legitimate postings, predicting everything as legitimate scores 95% accuracy but catches zero fraud. We use precision, recall, F1, confusion matrix, threshold analysis, and ROC-AUC.

In [ ]:
model.eval()
all_preds, all_probs, all_true = [], [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs  = F.softmax(logits, dim=1)[:, 1]
        preds  = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_true  = np.array(all_true)

print('=' * 55)
print('CLASSIFICATION REPORT — Validation Set')
print('=' * 55)
print(classification_report(all_true, all_preds, target_names=['Legitimate', 'Fraudulent'], zero_division=0))
print(f'ROC-AUC: {roc_auc_score(all_true, all_probs):.4f}')

In [ ]:
cm = confusion_matrix(all_true, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraudulent']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — BERT (Validation Set)')

axes[1].axis('off')
table_data = [
    ['Metric', 'Count', 'Meaning'],
    ['True Positives (TP)',  tp, 'Fraud correctly flagged'],
    ['True Negatives (TN)',  tn, 'Legit correctly cleared'],
    ['False Positives (FP)', fp, 'Legit wrongly flagged (false alarm)'],
    ['False Negatives (FN)', fn, 'Fraud missed — most costly error'],
    ['Precision', f'{tp/(tp+fp+1e-9):.3f}', 'How accurate our fraud flags are'],
    ['Recall',    f'{tp/(tp+fn+1e-9):.3f}', 'How much fraud we actually catch'],
]
t = axes[1].table(cellText=table_data[1:], colLabels=table_data[0], loc='center', cellLoc='left')
t.auto_set_font_size(True)
t.scale(1, 1.6)
axes[1].set_title('FP vs FN Breakdown', pad=20)
plt.suptitle('Confusion Matrix Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'False Positives: {fp} — legitimate jobs incorrectly flagged')
print(f'False Negatives: {fn} — fraudulent jobs missed (directly harms job seekers)')

In [ ]:
# ── FP vs FN Tradeoff across decision thresholds ─────────────────────────────
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
rows = []
for thresh in thresholds:
    preds_t = (all_probs >= thresh).astype(int)
    cm_t = confusion_matrix(all_true, preds_t, labels=[0, 1])
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    prec = tp_t / (tp_t + fp_t + 1e-9)
    rec  = tp_t / (tp_t + fn_t + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    rows.append({'Threshold': thresh, 'TP': tp_t, 'FP': fp_t, 'FN': fn_t,
                 'Precision': round(prec, 3), 'Recall': round(rec, 3), 'F1': round(f1, 3)})

print('FP vs FN Tradeoff at Different Decision Thresholds:')
print(pd.DataFrame(rows).to_string(index=False))
print('\nLower threshold = more fraud caught (higher recall) but more false alarms.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(all_true, all_probs)
auc = roc_auc_score(all_true, all_probs)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'BERT (AUC = {auc:.3f})')
axes[0].plot([0,1],[0,1],'gray',linestyle='--',label='Random baseline')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
prec_vals, rec_vals, pr_thresh = precision_recall_curve(all_true, all_probs)
axes[1].plot(rec_vals, prec_vals, color='tomato', lw=2)
if len(pr_thresh) > 0:
    idx = np.argmin(np.abs(pr_thresh - 0.5))
    if idx < len(rec_vals) - 1:
        axes[1].scatter(rec_vals[idx], prec_vals[idx], color='black', s=80, zorder=5, label='Threshold=0.5')
        axes[1].legend()
axes[1].set_xlabel('Recall (Fraud Detection Rate)')
axes[1].set_ylabel('Precision (Accuracy of Fraud Flags)')
axes[1].set_title('Precision-Recall Curve')
axes[1].grid(True, alpha=0.3)

plt.suptitle('BERT — Performance Curves on Validation Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary

| Deliverable | Detail |
|---|---|
| Training data | `FINAL_AUGMENTED_TRAINING.csv` (SMOTENC-balanced) |
| Model | `bert-base-uncased` fine-tuned, 3 epochs, LR=2e-5 |
| Saved weights | `models/nlp_bert.pth` (Drive + local) |
| Inference | `src/nlp_stream.predict_proba_from_row(row)` |
| Evaluation | Confusion matrix · threshold sweep · ROC-AUC · PR curve |

**Next:** Fusion Layer — combine `nlp_stream.predict_proba_from_row()` + `outlier_stream.anomaly_score()` via XGBoost in `src/main.py`.